[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/data_analysis_assistant/langgraph.ipynb)

# Data-analysis assistant — LangGraph

**Task.** Produce a supported household-waste summary while LangGraph makes each safe transition visible.

**Read this notebook as:** choose a provider → load data → declare graph nodes → execute and validate the graph.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "analysis-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [1]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [2]:
import csv
import json
from pathlib import Path
from typing import ClassVar, Literal

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import AnalysisRequest, file_sha256, summarise_reduction  # noqa: E402

data_path = ROOT / "data/data_analysis_assistant/household_waste.csv"
expected = json.loads((ROOT / "data/data_analysis_assistant/expected_summary.json").read_text())
fixture_path = ROOT / "data/data_analysis_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Graph and deterministic execution boundary
The model never emits code or computed values. Conditional routing rejects unknown tools before execution; schema and oracle mismatches stop before reporting.

In [3]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Decision(Strict):
    schema_id: ClassVar[str] = "analysis.decision.v1"
    tool: Literal["mean_reduction_by_intervention"]
    group_column: Literal["intervention"]
    before_column: Literal["before_kg"]
    after_column: Literal["after_kg"]


class Report(Strict):
    schema_id: ClassVar[str] = "analysis.report.v1"
    report: str
    groups: tuple[str, ...]


Decision.model_rebuild(_types_namespace={"Literal": Literal})


def fresh_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def build_graph():
    client = fresh_model()

    def inspect_node(state):
        with data_path.open(newline="", encoding="utf-8") as handle:
            rows = list(csv.DictReader(handle))
        return {
            "rows": len(rows),
            "columns": tuple(rows[0]) if rows else (),
            "trace": [{"event": "inspect"}],
            "model_calls": 0,
        }

    async def decide_node(state):
        decision = await propose(
            client,
            Decision,
            f"For columns {state['columns']}, use tool "
            "mean_reduction_by_intervention with intervention, before_kg and after_kg.",
        )
        return {
            **state,
            "decision": decision,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "decision", "tool": decision.tool}],
        }

    def execute_node(state):
        request = AnalysisRequest(
            group_column=state["decision"].group_column,
            before_column=state["decision"].before_column,
            after_column=state["decision"].after_column,
        )
        result = summarise_reduction(data_path, request)
        valid = result == expected["mean_reduction_kg"]
        return {
            **state,
            "result": result,
            "valid": valid,
            "trace": [*state["trace"], {"event": "execute_and_validate", "valid": valid}],
        }

    async def report_node(state):
        report = await propose(
            client, Report, f"Report only {state['result']}; include every key once in groups."
        )
        groups_valid = set(report.groups) == set(state["result"])
        return {
            **state,
            "report": report,
            "outcome": "supported_report" if groups_valid else "fallback",
            "model_calls": 2,
            "termination": "criteria_met" if groups_valid else "validation_failed",
            "trace": [*state["trace"], {"event": "report_validation", "valid": groups_valid}],
        }

    graph = StateGraph(dict)
    graph.add_node("inspect", inspect_node)
    graph.add_node("decide", decide_node)
    graph.add_node("execute", execute_node)
    graph.add_node("report", report_node)
    graph.add_edge(START, "inspect")
    graph.add_edge("inspect", "decide")
    graph.add_conditional_edges(
        "decide",
        lambda s: "execute" if s["decision"].tool == "mean_reduction_by_intervention" else END,
        {"execute": "execute", END: END},
    )
    graph.add_conditional_edges(
        "execute", lambda s: "report" if s["valid"] else END, {"report": "report", END: END}
    )
    graph.add_edge("report", END)
    return graph.compile()


# --- Execution: run the workflow and evaluate its observable result ---
first = await build_graph().ainvoke({})
second = await build_graph().ainvoke({}) if MODEL_PROVIDER == "mock" else first
try:
    AnalysisRequest(group_column="missing", before_column="before_kg", after_column="after_kg")
    schema_mismatch_rejected = False
except ValueError:
    schema_mismatch_rejected = True
failure_demonstrations = {
    "invalid_tool": "conditional edge ends",
    "unsafe_code": "never evaluated",
    "schema_mismatch": schema_mismatch_rejected,
    "result_mismatch": "report edge not taken",
}
evaluation = {
    "component": first["result"] == expected["mean_reduction_kg"],
    "trajectory": len(first["trace"]) == 4 and first["model_calls"] <= 2,
    "task": first["outcome"] == "supported_report",
    "safety": first["decision"].tool == "mean_reduction_by_intervention"
    and schema_mismatch_rejected,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "outputs": first["result"],
    "resource_report": {"model_calls": first["model_calls"], "graph_nodes": 4},
    "failure_demonstrations": failure_demonstrations,
    "provenance": {"sha256": file_sha256(data_path)},
    "fallback": "invalid route or result ends before report",
}

{'evaluation': {'component': True,
  'trajectory': True,
  'task': True,
  'safety': True,
  'repeated_run': True},
 'trace': [{'event': 'inspect'},
  {'event': 'decision', 'tool': 'mean_reduction_by_intervention'},
  {'event': 'execute_and_validate', 'valid': True},
  {'event': 'report_validation', 'valid': True}],
 'outputs': {'information_only': 0.0,
  'meal_planning': 0.8,
  'smaller_plates': 0.85},
 'resource_report': {'model_calls': 2, 'graph_nodes': 4},
 'failure_demonstrations': {'invalid_tool': 'conditional edge ends',
  'unsafe_code': 'never evaluated',
  'schema_mismatch': True,
  'result_mismatch': 'report edge not taken'},
 'provenance': {'sha256': '57e141c0f4f694aaaec4fbb1aada3f50bfa74eb5d8bf355dc8b58d9e637f779a'},
 'fallback': 'invalid route or result ends before report'}

## Limitation
Graph control flow cannot make this descriptive dataset causal; arbitrary model-authored analysis remains deliberately unsupported.